# Assessment 01 — Python & pandas Fundamentals

| | |
|---|---|
| **Estimated time** | 15–20 minutes (self-timed) |
| **Skill level tested** | Python fundamentals · pandas · NumPy · statistical thinking |
| **Prerequisites** | `pandas`, `numpy` — no external data files required |

---

## Instructions

Work through the five exercises below **without consulting documentation unless you are
genuinely stuck**. The goal is to surface what you already know fluently vs. what
you need to look up — both are useful signal.

Each exercise has:
- A **problem statement** (markdown cell)
- A **code skeleton** — fill in the `# YOUR CODE HERE` sections
- A **collapsed hint** cell you can open if needed (expand via the cell toolbar)

> **Tip — SAP context:** The sample data uses SAP-like field names (`MATNR`,
> `MTART`, `WERKS`) so the domain feels familiar. The Python/pandas patterns
> tested here map directly to analytics work on SAP extracts.

Run the setup cell first, then work top-to-bottom.


In [ ]:
# Setup — run this cell first
import pandas as pd
import numpy as np
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("\n✓ Environment ready")


---

## Exercise 1 — Deduplicate & Count Material Numbers

**Scenario:** A downstream report dumps raw material numbers extracted from
MM60 with duplicates.  You need a clean, sorted list plus a frequency count for
audit purposes.

**Task:**  
Write a function `clean_materials(material_list)` that:
1. Returns a **deduplicated, sorted** list of material numbers.
2. Returns a **dict** mapping each unique material number → occurrence count.

The function should return **both** as a tuple `(sorted_list, count_dict)`.

**Example input:**
```python
raw = ["ROH-1001", "HALB-2002", "ROH-1001", "FERT-3003",
       "HALB-2002", "HALB-2002", "FERT-3003", "ROH-1001"]
```

**Expected output (sorted_list):**  
`['FERT-3003', 'HALB-2002', 'ROH-1001']`

**Expected output (count_dict):**  
`{'FERT-3003': 2, 'HALB-2002': 3, 'ROH-1001': 3}`

*Tests: basic Python, list comprehension / dict, sorting.*


In [ ]:
%%time

raw_materials = [
    "ROH-1001", "HALB-2002", "ROH-1001", "FERT-3003",
    "HALB-2002", "HALB-2002", "FERT-3003", "ROH-1001",
    "NLAG-4004", "NLAG-4004",
]

def clean_materials(material_list):
    """
    Parameters
    ----------
    material_list : list of str

    Returns
    -------
    tuple : (sorted_unique_list, count_dict)
    """
    # YOUR CODE HERE
    pass


# ── Test your function ──────────────────────────────────────────────────────
sorted_list, count_dict = clean_materials(raw_materials)

print("Sorted unique list:", sorted_list)
print("\nCount dict:")
for mat, cnt in count_dict.items():
    print(f"  {mat}: {cnt}")

# Quick assertions (uncomment to validate)
# assert sorted_list == ['FERT-3003', 'HALB-2002', 'NLAG-4004', 'ROH-1001']
# assert count_dict['ROH-1001'] == 3
# assert count_dict['HALB-2002'] == 3
# print("\n✓ All assertions passed")


<details>
<summary>💡 Hint (click to expand)</summary>

- `dict` / `collections.Counter` both work for counting.
- `sorted(set(...))` handles deduplication + sorting in one line.
- A dict comprehension `{m: material_list.count(m) for m in set(material_list)}`
  is readable but O(n²) — `Counter` is O(n). Either is fine at this scale.

</details>


---

## Exercise 2 — Inventory Value per Material Type

**Scenario:** You received a CSV extract from MB52 (warehouse stocks). Each row is
a material/plant combination with quantity and unit price.

**Task:**  
1. Load the embedded CSV data using `pd.read_csv(StringIO(...))`.
2. Calculate **total inventory value** (`QTY * UNIT_PRICE`) per `MTART`
   (material type).
3. Return a **DataFrame** sorted descending by total inventory value, with
   columns `MTART` and `TOTAL_VALUE`.

*Tests: `pd.read_csv`, `groupby`, `agg`, `sort_values`.*


In [ ]:
%%time

MB52_CSV = """MATNR,MTART,WERKS,QTY,UNIT_PRICE
100-100,ROH,1000,500,12.50
100-101,ROH,1000,300,8.75
100-102,HALB,1000,200,45.00
100-103,HALB,1000,150,62.30
100-104,FERT,1000,80,125.00
100-105,FERT,2000,60,135.50
100-106,ROH,2000,400,9.20
100-107,NLAG,1000,1000,2.10
100-108,NLAG,2000,750,3.30
100-109,FERT,2000,45,198.00
"""

def inventory_value_by_type(csv_data):
    """
    Parameters
    ----------
    csv_data : str — raw CSV text

    Returns
    -------
    pd.DataFrame with columns [MTART, TOTAL_VALUE], sorted descending
    """
    # YOUR CODE HERE
    # 1. Read CSV
    # 2. Calculate INV_VALUE = QTY * UNIT_PRICE
    # 3. Group by MTART, sum INV_VALUE
    # 4. Sort and rename columns
    pass


result = inventory_value_by_type(MB52_CSV)
print(result.to_string(index=False))

# Quick assertions
# assert result.columns.tolist() == ['MTART', 'TOTAL_VALUE']
# assert result.iloc[0]['MTART'] in ['FERT', 'ROH']   # high-value types
# print("\n✓ All assertions passed")


<details>
<summary>💡 Hint (click to expand)</summary>

```python
df = pd.read_csv(StringIO(csv_data))
df['INV_VALUE'] = df['QTY'] * df['UNIT_PRICE']
result = (
    df.groupby('MTART')['INV_VALUE']
      .sum()
      .reset_index()
      .rename(columns={'INV_VALUE': 'TOTAL_VALUE'})
      .sort_values('TOTAL_VALUE', ascending=False)
)
```

</details>


---

## Exercise 3 — Flag At-Risk Open Orders (Vectorized)

**Scenario:** Your order pipeline has stale open orders that should be escalated.
An order is **AT_RISK** if its `STATUS` is `'OPEN'` **and** the `ORDER_DATE` is
more than 60 days in the past.  Any other order keeps its original status string.

**Task:**  
Write a function `flag_at_risk(df)` that adds a new column `RISK_FLAG` and
returns the DataFrame.  
- **No Python loops** (`for`, `while`, `iterrows`) — use vectorized pandas /
  NumPy operations.
- Use `np.where` or `pd.Series.where` / `mask`.

*Tests: datetime arithmetic, boolean indexing, vectorized operations.*


In [ ]:
%%time
import numpy as np
from datetime import datetime, timedelta

# Sample orders — some dates are deliberately old
today = pd.Timestamp.today().normalize()
orders_data = {
    'ORDER_ID':   ['SO-001','SO-002','SO-003','SO-004','SO-005',
                   'SO-006','SO-007','SO-008'],
    'MATNR':      ['100-100','100-101','100-102','100-103','100-104',
                   '100-105','100-106','100-107'],
    'STATUS':     ['OPEN','DELIVERED','OPEN','CANCELLED','OPEN',
                   'OPEN','DELIVERED','OPEN'],
    'ORDER_DATE': [
        today - timedelta(days=90),   # old OPEN  → AT_RISK
        today - timedelta(days=30),   # DELIVERED → no change
        today - timedelta(days=10),   # recent OPEN → no change
        today - timedelta(days=120),  # CANCELLED   → no change
        today - timedelta(days=65),   # old OPEN  → AT_RISK
        today - timedelta(days=5),    # recent OPEN → no change
        today - timedelta(days=200),  # DELIVERED → no change
        today - timedelta(days=75),   # old OPEN  → AT_RISK
    ]
}
orders_df = pd.DataFrame(orders_data)
orders_df['ORDER_DATE'] = pd.to_datetime(orders_df['ORDER_DATE'])


def flag_at_risk(df, days_threshold=60):
    """
    Parameters
    ----------
    df : pd.DataFrame with columns [ORDER_ID, MATNR, STATUS, ORDER_DATE]
    days_threshold : int, default 60

    Returns
    -------
    pd.DataFrame with new column RISK_FLAG:
        'AT_RISK' if STATUS=='OPEN' and ORDER_DATE < today - days_threshold
        else original STATUS value
    """
    df = df.copy()
    # YOUR CODE HERE
    pass


result = flag_at_risk(orders_df)
print(result[['ORDER_ID', 'STATUS', 'ORDER_DATE', 'RISK_FLAG']].to_string(index=False))

# Quick assertions
# at_risk = result[result['RISK_FLAG'] == 'AT_RISK']['ORDER_ID'].tolist()
# assert set(at_risk) == {'SO-001', 'SO-005', 'SO-008'}
# print("\n✓ All assertions passed")


<details>
<summary>💡 Hint (click to expand)</summary>

```python
cutoff = pd.Timestamp.today() - pd.Timedelta(days=days_threshold)
condition = (df['STATUS'] == 'OPEN') & (df['ORDER_DATE'] < cutoff)
df['RISK_FLAG'] = np.where(condition, 'AT_RISK', df['STATUS'])
```

`np.where(cond, value_if_true, value_if_false)` is the vectorized equivalent
of an if/else applied element-wise.

</details>


---

## Exercise 4 — Merge & Pivot: Revenue by Material Type × Status

**Scenario:** You have two DataFrames — a materials master and a sales order
line-item extract.  Your manager wants a cross-tab of **total revenue** by
`MATERIAL_TYPE` (rows) and `STATUS` (columns).

**Task:**  
1. Merge the two DataFrames on `MATNR` (left join from `sales_orders`).
2. Calculate `REVENUE = QTY_ORDERED * UNIT_PRICE`.
3. Build a `pivot_table` with `MATERIAL_TYPE` as index, `STATUS` as columns,
   values = sum of `REVENUE`.
4. `fillna(0)` and round to 2 decimal places.
5. Return the pivot DataFrame.

*Tests: `pd.merge`, `pivot_table`, `fillna`, handling missing join keys.*


In [ ]:
%%time

materials_master = pd.DataFrame({
    'MATNR':         ['100-100','100-101','100-102','100-103','100-104',
                      '100-105','100-106','100-107','100-108'],
    'MATERIAL_TYPE': ['ROH','ROH','HALB','HALB','FERT',
                      'FERT','ROH','NLAG','NLAG'],
    'UNIT_PRICE':    [12.50, 8.75, 45.00, 62.30, 125.00,
                      135.50, 9.20, 2.10, 3.30],
})

sales_orders = pd.DataFrame({
    'ORDER_ID':   ['SO-001','SO-002','SO-003','SO-004','SO-005',
                   'SO-006','SO-007','SO-008','SO-009','SO-010',
                   'SO-011','SO-012'],
    'MATNR':     ['100-100','100-100','100-102','100-103','100-104',
                  '100-104','100-105','100-101','100-106','100-999',  # 100-999 not in master
                  '100-107','100-108'],
    'QTY_ORDERED':[50, 30, 20, 15, 10, 8, 12, 40, 60, 5, 200, 150],
    'STATUS':    ['DELIVERED','OPEN','DELIVERED','CANCELLED','DELIVERED',
                  'OPEN','DELIVERED','DELIVERED','OPEN','DELIVERED',
                  'CANCELLED','OPEN'],
})


def revenue_pivot(materials_df, orders_df):
    """
    Returns a pivot_table: MATERIAL_TYPE × STATUS → sum(REVENUE)
    Missing combos filled with 0. Values rounded to 2 dp.
    """
    # YOUR CODE HERE
    # 1. merge
    # 2. calculate REVENUE
    # 3. pivot_table
    # 4. fillna + round
    pass


pivot = revenue_pivot(materials_master, sales_orders)
print(pivot)
print("\nShape:", pivot.shape)


<details>
<summary>💡 Hint (click to expand)</summary>

```python
merged = orders_df.merge(materials_df, on='MATNR', how='left')
merged['REVENUE'] = merged['QTY_ORDERED'] * merged['UNIT_PRICE']
pivot = merged.pivot_table(
    index='MATERIAL_TYPE',
    columns='STATUS',
    values='REVENUE',
    aggfunc='sum'
).fillna(0).round(2)
```

Rows where `MATNR` is not in `materials_df` will have `NaN` for `UNIT_PRICE`
and therefore `NaN` for `REVENUE` — worth noting in a real analysis.

</details>


---

## Exercise 5 — IQR Outlier Detection Function

**Scenario:** Before feeding data into a reporting layer, you want a reusable
function to flag statistical outliers in any numeric column.

**Task:**  
Write `detect_outliers(df, column)` that:
1. Calculates Q1, Q3, and IQR for `df[column]`.
2. Defines bounds:  
   - `lower_bound = Q1 − 1.5 × IQR`  
   - `upper_bound = Q3 + 1.5 × IQR`
3. Returns a **dict** with keys:
   - `outlier_count` — number of outlier rows
   - `outlier_pct` — percentage of total rows (2 dp)
   - `lower_bound` — float (2 dp)
   - `upper_bound` — float (2 dp)
   - `outlier_values` — list of the actual outlier values, sorted

*Tests: statistical thinking, `.quantile()`, clean function design.*


In [ ]:
%%time
import numpy as np

# Simulate a unit price series with a few pricing errors
np.random.seed(42)
normal_prices = np.random.normal(loc=50, scale=8, size=95)   # realistic prices
outlier_prices = np.array([0.01, 0.05, 250.0, 380.0, 999.0])  # obvious errors
all_prices = np.concatenate([normal_prices, outlier_prices])
np.random.shuffle(all_prices)

price_df = pd.DataFrame({'MATNR': [f'M-{i:04d}' for i in range(100)],
                          'UNIT_PRICE': all_prices.round(2)})


def detect_outliers(df, column):
    """
    IQR-based outlier detector.

    Parameters
    ----------
    df     : pd.DataFrame
    column : str — name of numeric column to analyse

    Returns
    -------
    dict with keys: outlier_count, outlier_pct, lower_bound,
                    upper_bound, outlier_values
    """
    # YOUR CODE HERE
    pass


result = detect_outliers(price_df, 'UNIT_PRICE')

print("Outlier Report — UNIT_PRICE")
print(f"  Count        : {result['outlier_count']}")
print(f"  Percentage   : {result['outlier_pct']}%")
print(f"  Lower bound  : {result['lower_bound']}")
print(f"  Upper bound  : {result['upper_bound']}")
print(f"  Values       : {result['outlier_values']}")

# Quick assertions
# assert result['outlier_count'] == 5
# assert result['outlier_pct'] == 5.0
# print("\n✓ All assertions passed")


<details>
<summary>💡 Hint (click to expand)</summary>

```python
series = df[column].dropna()
q1 = series.quantile(0.25)
q3 = series.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
mask = (series < lower) | (series > upper)
outliers = series[mask]
return {
    'outlier_count':  int(mask.sum()),
    'outlier_pct':    round(100 * mask.mean(), 2),
    'lower_bound':    round(lower, 2),
    'upper_bound':    round(upper, 2),
    'outlier_values': sorted(outliers.tolist()),
}
```

</details>


---

## Self-Scoring Rubric

Score yourself on each exercise:

| # | Topic | Solved fluently (no hints) | Solved with hint | Couldn't solve |
|---|-------|:--------------------------:|:----------------:|:--------------:|
| 1 | Python: dedup + count | ☐ | ☐ | ☐ |
| 2 | pandas: read_csv + groupby | ☐ | ☐ | ☐ |
| 3 | Vectorized datetime logic | ☐ | ☐ | ☐ |
| 4 | Merge + pivot_table | ☐ | ☐ | ☐ |
| 5 | IQR outlier function | ☐ | ☐ | ☐ |

### Interpretation

| Score | Meaning | Suggested next step |
|-------|---------|---------------------|
| 5 × Fluent | Strong foundation | Move directly to SQL & combined challenge |
| 3–4 × Fluent | Solid — minor gaps | Review pandas `.groupby`, `.merge`, and `np.where` patterns |
| 1–2 × Fluent | Needs structured refresh | Work through pandas 10-minute tutorial, then retry |
| 0 × Fluent | Fundamentals needed | Complete a pandas basics module first |

> **Time check:** If you took > 30 minutes, note which exercises caused the
> most friction — those topics are your priority review areas.
